In [1]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

generator = pipeline(task = "text-generation",
                     model = "gpt2",
                     device = device)
outputs = generator(text_inputs = "Machine learning is",
                    max_length = 20,
                    num_return_sequences = 3,
                    pad_token_id = generator.tokenizer.eos_token_id)
print(outputs)

cuda


c:\Users\use\Documents\KDT17\cv-deep-learning\.venv-torchtext\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': "Machine learning is now being used on more people, and I don't believe that there's anything wrong"}, {'generated_text': 'Machine learning is all about iteratively iterating over new variables. This is much easier than iterating'}, {'generated_text': 'Machine learning is an innovative approach to the integration of machine learning technology through a deep neural network framework.'}]


In [2]:
# Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
# [{'generated_text': 'Machine learning is now used as a bridge between the physical and mental sciences, while the two disciplines are'}, {'generated_text': 'Machine learning is done by the machine learning software.\n\n"We can get much better at it'}, {'generated_text': "Machine learning is what you're doing and it's only going to make things worse. And this is"}]

In [3]:

# Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
# [{'generated_text': "Machine learning is still a relatively new field in physics, and even though it's still relatively new,"}, {'generated_text': 'Machine learning is a real challenge.\n\nThe most simple way is to create a task. The'}, {'generated_text': 'Machine learning is an easy way to add value to your customers in every way the ability and cost effective'}]

In [4]:
# Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
# [{'generated_text': 'Machine learning is now on the rise, and many think that it only works if all of the inputs'}, {'generated_text': 'Machine learning is about creating an optimal model that fits the specific data sets presented in a given paradigm.'}, {'generated_text': 'Machine learning is not simple work that requires the ability to learn and apply algorithms. It is, therefore'}]

In [5]:
# bert 사용

import numpy as np
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
df = pd.DataFrame(corpus.test).sample(20000, random_state = 42)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\use\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\use\K

In [6]:
train, valid, test = np.split(df.sample(frac = 1, random_state = 42),
                              [int(0.6 * len(df)),
                               int(0.8 * len(df))])

print(train.head().to_markdown())
print(len(train))
print(len(valid))
print(len(test))

|       | text                                                     |   label |
|------:|:---------------------------------------------------------|--------:|
| 26891 | 역시 코믹액션은 성룡, 홍금보, 원표 삼인방이 최고지!!     |       1 |
| 25024 | 점수 후하게 줘야것네 별 반개~                            |       0 |
| 11666 | 오랜만에 느낄수 있는 [감독] 구타욕구.                    |       0 |
| 40303 | 본지는 좀 됬지만 극장서 돈주고 본게 아직까지 아까운 영화 |       0 |
| 18010 | 징키스칸이란 소재를 가지고 이것밖에 못만드냐             |       0 |
12000
4000
4000


c:\Users\use\Documents\KDT17\cv-deep-learning\.venv-torchtext\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [7]:
import torch
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(text = data.text.tolist(),
                          padding = "longest",
                          truncation = True,
                          return_tensors = "pt")
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    labels = torch.tensor(data.label.values, dtype = torch.long).to(device)
    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset,
                            sampler = data_sampler,
                            batch_size = batch_size)
    return dataloader

epochs = 5
batch_size = 32

tokenizer = BertTokenizer.from_pretrained(pretrained_model_name_or_path = "bert-base-multilingual-cased",
                                          do_lower_case = False)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

(tensor([   101,  58466,   9812, 118956, 119122,  59095,  10892,   9434, 118888,
           117,   9992,  40032,  30005,    117,   9612,  37824,   9410,  12030,
         42337,  10739,  83491,  12508,    106,    106,    102,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,

In [8]:
from torch import optim
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(pretrained_model_name_or_path = "bert-base-multilingual-cased",
                                                      num_labels = 2).to(device)
optimizer = optim.AdamW(model.parameters(), lr = 1e-5, eps = 1e-8)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---", sssub_name)

bert
- embeddings
-- word_embeddings
-- position_embeddings
-- token_type_embeddings
-- LayerNorm
-- dropout
- encoder
-- layer
--- 0
--- 1
--- 2
--- 3
--- 4
--- 5
--- 6
--- 7
--- 8
--- 9
--- 10
--- 11
- pooler
-- dense
-- activation
dropout
classifier


In [10]:
# train을 위한 함수 생성

import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis = 1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids = input_ids,
                        attention_mask = attention_mask,
                        labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids = input_ids,
                            attention_mask = attention_mask,
                            labels = labels)
            logits = outputs.logits
            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss.item()
            val_accuracy += accuracy
    val_loss = val_loss / len(dataloader)
    val_accuracy = val_accuracy / len(dataloader)
    return val_loss, val_accuracy

In [ ]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(epoch + 1, train_loss, val_loss, val_accuracy)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "models/BertForSequenceClassification.pt")
        print("Saved!!")

1 0.5605732629299164 0.45328997254371645 0.7835
Saved!!


In [ ]:
model = BertForSequenceClassification.from_pretrained(pretrained_model_name_or_path = "bert-base-multilingual-cased",
                                                      num_labels = 2).to(device)
model.load_state_dict(torch.load("models/BertForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(test_loss)
print(test_accuracy)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0.43164048808813094
0.80275


In [ ]:
# Bart 사용 / 뉴스요약
import numpy as np

from datasets import load_dataset

news = load_dataset("argilla/news-summary", split = "test")
df = news.to_pandas().sample(5000, random_state = 42)[["text", "prediction"]]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

train, valid, test = np.split(df.sample(frac = 1,
                                        random_state = 42),
                              [int(0.6 * len(df)),
                               int(0.8 * len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

README.md: 0.00B [00:00, ?B/s]

c:\Users\use\Documents\KDT17\cv-deep-learning\.venv-torchtext\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\use\.cache\huggingface\hub\datasets--argilla--news-summary. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


NameError: name 'np' is not defined

In [ ]:
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(text = data.text.tolist(),
                          padding = "longest",
                          truncation = True,
                          return_tensors = "pt",
                          max_length = 1024)
    labels = []
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors = "pt").squeeze())
    labels = pad_sequence(labels,
                          batch_first = True,
                          padding_value = -100).to(device)

    return TensorDataset(input_ids, attention_mask, labels)

def get_dataloader(dataset,
                   sampler,
                   batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset,
                            sampler = data_sampler,
                            batch_size = batch_size)
    return dataloader

epochs = 5
batch_size = 8

tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path = "facebook/bart-base")

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

vocab.json: 0.00B [00:00, ?B/s]

c:\Users\KDT-17\Documents\17-NLP & CV\.venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KDT-17\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

c:\Users\KDT-17\Documents\17-NLP & CV\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


(tensor([   0,  495, 1889,  ...,    1,    1,    1], device='cuda:0'), tensor([1, 1, 1,  ..., 0, 0, 0], device='cuda:0'), tensor([    0, 35891,   161,    56,  5616, 10405,    19,   140,    23,  5490,
         3564,     2,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
       device='cuda:0'))


In [ ]:
from torch import optim
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path = "facebook/bart-base").to(device)
optimizer = optim.AdamW(model.parameters(), lr = 5e-5, eps = 1e-8) # AdamW 는 eps(입실론값)

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [ ]:
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("-", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("--", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("---", sssub_name)

model
- shared
- encoder
-- embed_tokens
-- embed_positions
-- layers
--- 0
--- 1
--- 2
--- 3
--- 4
--- 5
-- layernorm_embedding
- decoder
-- embed_tokens
-- embed_positions
-- layers
--- 0
--- 1
--- 2
--- 3
--- 4
--- 5
-- layernorm_embedding
lm_head


In [ ]:
# 요약시 Bart를 사용하면 전용 점수로 계산

import evaluate

rouge_score = evaluate.load("rouge")

def calc_rouge(preds, labels):
    preds = preds.argmax(axis = -1)
    labels = np.where(labels != -100,
                      labels,
                      tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens = True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens = True)

    rouge2 = rouge_score.compute(predictions = decoded_preds,
                                 references = decoded_labels)

    return rouge2["rouge2"]

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(input_ids = input_ids,
                        attention_mask = attention_mask,
                        labels = labels)
        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss, val_rouge = 0.0, 0.0
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(input_ids = input_ids,
                            attention_mask = attention_mask,
                            labels = labels)
            logits = outputs.logits
            loss = outputs.loss

            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            rouge = calc_rouge(logits, label_ids)

            val_loss += loss.item()
            val_rouge += rouge

    val_loss = val_loss / len(dataloader)
    val_rouge = val_rouge / len(dataloader)
    return val_loss, val_rouge

In [ ]:
rouge_score = evaluate.load("rouge", tokenizer = tokenizer)
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_rouge = evaluation(model, valid_dataloader)
    print(epoch + 1, train_loss, val_loss, val_rouge)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "models/BartForConditionalGeneration.pt")
        print("Saved!!")

In [ ]:
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path = "facebook/bart-base").to(device)
model.load_state_dict(torch.load("models/BartForConditionalGeneration.pt"))

test_loss, test_rouge = evaluation(model, test_dataloader)
print(test_loss)
print(test_rouge)

In [ ]:
from transformers import pipeline

summarizer = pipeline(task = 'summarization', model = model, tokenizer = tokenizer, max_length = 54, device = device)

for index in range(5):
    news_text = test.text.iloc[index]
    summarization = test.prediction.iloc[index]
    predicted_summarization = summarizer(news_text)[0]['summary_text']
    print(summarization)
    print(predicted_summarization)